In [1]:
import pandas as pd
import os
import numpy as np
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
# definindo os caminhos
path = os.getenv('DATASETS_PATH')
data_path = os.getenv('DATA_PATH')

# carregando dataset de appearances
df = pd.read_csv(path + '/appearances.csv')
df.head()

,appearance_id,game_id,player_id,player_club_id,player_current_club_id,date,player_name,competition_id,yellow_cards,red_cards,goals,assists,minutes_played
0,2231978_38004,2231978,38004,853,235,2012-07-03,Aurélien Joachim,CLQ,0,0,2,0,90
1,2233748_79232,2233748,79232,8841,2698,2012-07-05,Ruslan Abyshov,ELQ,0,0,0,0,90
2,2234413_42792,2234413,42792,6251,465,2012-07-05,Sander Puri,ELQ,0,0,0,0,45
3,2234418_73333,2234418,73333,1274,76,2012-07-05,Vegar Hedenstad,ELQ,0,0,0,0,90
4,2234421_122011,2234421,122011,195,3008,2012-07-05,Markus Henriksen,ELQ,0,0,0,1,90


In [4]:
# vendo informações gerais do dataset
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1889406 entries, 0 to 1889405
Data columns (total 13 columns):
 #   Column                  Dtype 
---  ------                  ----- 
 0   appearance_id           object
 1   game_id                 int64 
 2   player_id               int64 
 3   player_club_id          int64 
 4   player_current_club_id  int64 
 5   date                    object
 6   player_name             object
 7   competition_id          object
 8   yellow_cards            int64 
 9   red_cards               int64 
 10  goals                   int64 
 11  assists                 int64 
 12  minutes_played          int64 
dtypes: int64(9), object(4)
memory usage: 187.4+ MB


In [5]:
# parse da data e cobertura temporal das aparições
df['date'] = pd.to_datetime(df['date'])
print("Primeiro registro:", df['date'].min())
print("Último registro:", df['date'].max())

Primeiro registro: 2012-07-03 00:00:00
Último registro: 2026-06-28 00:00:00


In [6]:
# vendo as diferentes ligas e seus volumes
df['competition_id'].value_counts()

competition_id
IT1     154999
ES1     153675
GB1     149685
FR1     143246
TR1     132007
L1      124511
NL1     119807
PO1     119627
BE1     100778
RU1      95218
GR1      87499
SC1      80954
UKR1     75861
DK1      65161
EL       55676
CL       49470
ELQ      19334
CDR      17310
GRP      15400
RUP      14691
FAC      13934
DFB      11338
CIT      11284
NLP      10523
CLQ      10483
POCP      9757
ECLQ      9584
DKP       9546
SFA       7388
UKRP      6542
CGB       2285
FIWC      2053
KLUB      1808
SUC        973
POBE       907
EJPL       905
AFCN       776
SCI        597
GBCS       415
DFL        411
USC        407
RUSS       406
FRCH       401
POSU       393
BESC       372
NLSC       368
BPO4       366
UKRS       275
Name: count, dtype: int64

In [7]:
# criando season_year — ano de início da temporada (temporada começa em agosto)
df['season_year'] = df['date'].apply(
    lambda d: d.year if d.month >= 8 else d.year - 1
)
df[['date', 'season_year']].head(10)

,date,season_year
0,2012-07-03,2011
1,2012-07-05,2011
2,2012-07-05,2011
3,2012-07-05,2011
4,2012-07-05,2011
5,2012-07-05,2011
6,2012-07-05,2011
7,2012-07-05,2011
8,2012-07-05,2011
9,2012-07-05,2011


In [8]:
# agregando stats por (player_id, season_year)
df_temp = df.groupby(['player_id', 'season_year'], as_index=False).agg(
    goals_season=('goals', 'sum'),
    assists_season=('assists', 'sum'),
    minutes_season=('minutes_played', 'sum'),
    games_season=('appearance_id', 'count'),
    yellow_cards_season=('yellow_cards', 'sum'),
    red_cards_season=('red_cards', 'sum')
)
df_temp.head()

,player_id,season_year,goals_season,assists_season,minutes_season,games_season,yellow_cards_season,red_cards_season
0,10,2012,16,3,2585,36,8,0
1,10,2013,8,5,2220,29,2,0
2,10,2014,16,9,2289,40,6,0
3,10,2015,8,8,1714,31,3,0
4,26,2012,0,0,4491,50,2,1


In [9]:
# calculando features derivadas com tratamento de divisão por zero
df_temp['goals_per_90'] = np.where(
    df_temp['minutes_season'] > 0,
    df_temp['goals_season'] / (df_temp['minutes_season'] / 90),
    0
)
df_temp['assists_per_90'] = np.where(
    df_temp['minutes_season'] > 0,
    df_temp['assists_season'] / (df_temp['minutes_season'] / 90),
    0
)
df_temp['minutes_per_game'] = np.where(
    df_temp['games_season'] > 0,
    df_temp['minutes_season'] / df_temp['games_season'],
    0
)
df_temp.head()

,player_id,season_year,goals_season,assists_season,minutes_season,games_season,yellow_cards_season,red_cards_season,goals_per_90,assists_per_90,minutes_per_game
0,10,2012,16,3,2585,36,8,0,0.557060,0.104449,71.805556
1,10,2013,8,5,2220,29,2,0,0.324324,0.202703,76.551724
2,10,2014,16,9,2289,40,6,0,0.629096,0.353866,57.225000
3,10,2015,8,8,1714,31,3,0,0.420070,0.420070,55.290323
4,26,2012,0,0,4491,50,2,1,0.000000,0.000000,89.820000


In [10]:
# vendo informações gerais do dataset agregado
df_temp.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 102293 entries, 0 to 102292
Data columns (total 11 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   player_id            102293 non-null  int64  
 1   season_year          102293 non-null  int64  
 2   goals_season         102293 non-null  int64  
 3   assists_season       102293 non-null  int64  
 4   minutes_season       102293 non-null  int64  
 5   games_season         102293 non-null  int64  
 6   yellow_cards_season  102293 non-null  int64  
 7   red_cards_season     102293 non-null  int64  
 8   goals_per_90         102293 non-null  float64
 9   assists_per_90       102293 non-null  float64
 10  minutes_per_game     102293 non-null  float64
dtypes: float64(3), int64(8)
memory usage: 8.6 MB


In [11]:
# verificando nulos no dataset final
df_temp.isna().sum()

player_id              0
season_year            0
goals_season           0
assists_season         0
minutes_season         0
games_season           0
yellow_cards_season    0
red_cards_season       0
goals_per_90           0
assists_per_90         0
minutes_per_game       0
dtype: int64

In [12]:
# salvando appearances_by_season.parquet
df_temp.to_parquet(data_path + '/appearances_by_season.parquet', index=False)